In [1]:
!pip install evaluate
!pip install rouge_score
!pip install sacrebleu
!pip install openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 20.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.8 MB/s eta 0:00:00a 0:00:01
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24936 sha256=17dcca9aa11a8e45866ec7b2c4d1cc1cebadb867721cf774aec6508620acda5b
  Stored in directory: /tmp/pip-ephem-wheel-cache-5_28l2pb/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 21.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
from evaluate import load
import time
import numpy as np
import unittest
import os

2024-07-10 12:56:57.531042: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
def calcular_metricas(reference_texts, generated_texts):
    bleu = load("bleu")
    rouge = load("rouge")
    meteor = load("meteor")
    
    metrics = {
        'BLEU': [],
        'ROUGE': [],
        'METEOR': [],
    }
    
    for ref, gen in zip(reference_texts, generated_texts):
         # Limpiar textos de referencia y generados
        bleu_score = bleu.compute(predictions=[gen], references=[[ref]])['bleu']
        rouge_score = rouge.compute(predictions=[gen], references=[ref])['rouge1']
        meteor_score = meteor.compute(predictions=[gen], references=[ref])['meteor']
        
        metrics['BLEU'].append(bleu_score)
        metrics['ROUGE'].append(rouge_score)
        metrics['METEOR'].append(meteor_score)
    
    return metrics

def generar_metricas(resultados, coleccion_relatos, ruta):
    results_list = []
    for gen_col in resultados.columns:
        for ref_col in coleccion_relatos.columns:
            generated_texts = resultados[gen_col].dropna().tolist()
            reference_texts = coleccion_relatos[ref_col].dropna().tolist()

            metrics = calcular_metricas(reference_texts, generated_texts)

            for i in range(len(generated_texts)):
                if i < len(reference_texts):
                 
                    results_list.append({
                        'Columna Modelo': gen_col,
                        'Columna Res-Manual': ref_col,
                        'Fila': i + 1,
                        'BLEU': metrics['BLEU'][i],
                        'ROUGE': metrics['ROUGE'][i],
                        'METEOR': metrics['METEOR'][i],
                    })
                
    results = pd.DataFrame(results_list)
    results.to_excel(ruta, index=False, engine='openpyxl')

def main():
    try:
        coleccion_relatos = pd.read_excel('/home/jovyan/data/ColecciónRelatos-Resueltos.xlsx')
        resultados_1 = pd.read_excel('/home/jovyan/data/resultados_Prompt_1.xlsx')
        #resultados_2 = pd.read_excel('/home/jovyan/data/resultados_Prompt_2.xlsx')
        #resultados_3 = pd.read_excel('/home/jovyan/data/resultados_Prompt_3.xlsx')
    except FileNotFoundError as e:
        print(f"Archivo no encontrado: {e}")
        raise
        
    generar_metricas(resultados_1, coleccion_relatos, "/home/jovyan/data/Metricas_Prompt_1.xlsx")
    #generar_metricas(resultados_2, coleccion_relatos, "/home/jovyan/data/Metricas_Prompt_2.xlsx")    
    #generar_metricas(resultados_3, coleccion_relatos, "/home/jovyan/data/Metricas_Prompt_3.xlsx")

    
    print("----------------------FIN--------------------------")

if __name__ == "__main__":
    main()


[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [12]:
class TestCalcularMetricas(unittest.TestCase):

    def setUp(self):
        self.reference_texts = [
            "Este es el primer texto de referencia.",
            "Este es el segundo texto de referencia.",
        ]
        self.generated_texts = [
            "Este es el primer texto generado.",
            "Este es el segundo texto generado.",
        ]

    def test_calcular_metricas(self):
        metrics = calcular_metricas(self.reference_texts, self.generated_texts)
        self.assertIn('BLEU', metrics)
        self.assertIn('ROUGE', metrics)
        self.assertIn('METEOR', metrics)
        #self.assertIn('TER', metrics)

        self.assertEqual(len(metrics['BLEU']), len(self.reference_texts))
        self.assertEqual(len(metrics['ROUGE']), len(self.reference_texts))
        self.assertEqual(len(metrics['METEOR']), len(self.reference_texts))
        #self.assertEqual(len(metrics['TER']), len(self.reference_texts))

        # Verificar que los puntajes están en el rango correcto
        for score in metrics['BLEU']:
            self.assertIsInstance(score, float)
            self.assertGreaterEqual(score, 0.0)
            self.assertLessEqual(score, 1.0)

        for score in metrics['ROUGE']:
            self.assertIsInstance(score, float)
            self.assertGreaterEqual(score, 0.0)
            self.assertLessEqual(score, 1.0)

        for score in metrics['METEOR']:
            self.assertIsInstance(score, float)
            self.assertGreaterEqual(score, 0.0)
            self.assertLessEqual(score, 1.0)

In [15]:
class TestGenerarMetricas(unittest.TestCase):

    def setUp(self):
        # Crear datos de prueba
        self.resultados = pd.DataFrame({
            'Modelo1': ["Texto generado 1.", "Texto generado 2."]
        })
        self.coleccion_relatos = pd.DataFrame({
            'Referencia1': ["Texto de referencia 1.", "Texto de referencia 2."]
        })
        self.ruta = 'resultados_metricas.xlsx'

    def tearDown(self):
        # Eliminar archivo de prueba después de cada prueba
        if os.path.exists(self.ruta):
            os.remove(self.ruta)

    def test_generar_metricas(self):
        # Llamar a la función generar_metricas
        generar_metricas(self.resultados, self.coleccion_relatos, self.ruta)
        
        # Leer el archivo Excel generado
        df_resultados = pd.read_excel(self.ruta, engine='openpyxl')
        
        # Verificar que el archivo tiene las columnas correctas
        columnas_esperadas = ['Columna Modelo', 'Columna Res-Manual', 'Fila', 'BLEU', 'ROUGE', 'METEOR']
        for columna in columnas_esperadas:
            self.assertIn(columna, df_resultados.columns)
        
        # Verificar que el número de filas es el esperado
        filas_esperadas = min(len(self.resultados), len(self.coleccion_relatos))
        self.assertEqual(len(df_resultados), filas_esperadas)

        # Verificar que los valores no sean nulos
        for columna in columnas_esperadas:
            self.assertFalse(df_resultados[columna].isnull().any())

In [16]:
if __name__ == '__main__':
    unittest.main(argv=[''], exit=False) 

[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
.[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
.
----------------------------------------------------------------------
Ran 2 tests in 7.329s

OK
